# Hiring Bias Audit

**Based on:** *AI in Modern Psychology*, Chapter 13.

Train a hiring-decision model on synthetic data and audit it for disparate impact.


## Learning Objectives

- Compute demographic-parity, equal-opportunity, and equalized-odds metrics.
- Visualize per-group disparities.
- Try a simple mitigation strategy.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from code.io_ed.hiring_bias_audit import audit
from data.generate_synthetic import generate_hiring
rng = np.random.default_rng(0)


## Build a model


In [ ]:
df = generate_hiring(5_000, seed=42)
features = ['years_experience', 'education_years', 'test_score', 'interview_score']
X = df[features]; y = df['hired']; protected = df['protected_group'].values
X_tr, X_te, y_tr, y_te, p_tr, p_te = train_test_split(X, y, protected, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
y_pred = model.predict(X_te)
print(f'Accuracy: {(y_pred == y_te).mean():.3f}')


## Audit


In [ ]:
result = audit(y_te.values, y_pred, p_te)
print('DPD:',   result['demographic_parity_difference'])
print('EOD:',   result['equal_opportunity_difference'])
print('EOdds:', result['equalized_odds_difference'])
result['report']


## A simple mitigation: re-balance by inverse-propensity weighting


In [ ]:
weights = np.where(p_tr == 1, (p_tr == 0).mean() / max((p_tr == 1).mean(), 1e-3), 1.0)
model2 = LogisticRegression(max_iter=500).fit(X_tr, y_tr, sample_weight=weights)
y_pred2 = model2.predict(X_te)
result2 = audit(y_te.values, y_pred2, p_te)
print('DPD after mitigation:', result2['demographic_parity_difference'])
result2['report']


## Ethical Considerations

- Fairness metrics are mutually incompatible; choose by context.
- Mitigation often trades accuracy for fairness — document the trade-off in the model card (`ethics_toolkit/model_card_template.md`).
- An audit alone does not absolve a deployer; ongoing monitoring is required.


## References

- Barocas, S., Hardt, M., & Narayanan, A. (2019). *Fairness and Machine Learning*.
